# 1 · Create the Vector Search Index

Databricks **Free Edition does not include Knowledge Assistant**, so this
notebook builds the document-retrieval backend with **Vector Search** instead.

The contract PDFs are **print-to-PDF documents whose body is image-based**, so
plain text extraction returns only page headers. Instead of hand-rolling OCR
and chunking, we use Databricks' managed document pipeline:

1. **`ai_parse_document`** — runs OCR + layout analysis on each PDF.
2. **`ai_prep_search`** — turns the parsed document into **search-ready chunks**
   (with a context-enriched `chunk_to_embed` field) optimized for RAG.

We write those chunks to a Delta table and sync it into a **Delta Sync index**
with managed embeddings (`databricks-gte-large-en`). The index is attached
directly to the Supervisor Agent in notebook `3` as a native Vector Search tool.

> **Runtime requirement:** `ai_prep_search` needs Databricks Runtime **18.2+**
> (or serverless compute with environment version **3+**). Free Edition serverless
> satisfies this by default.


### Define Parameters

Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.

| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. Default `workspace` is pre-provisioned — no catalog creation needed. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `VOLUME_NAME` | `contracts` | Volume containing the fictional contract PDFs. |
| `SOURCE_TABLE` | `contract_chunks` | Delta table (CDF enabled) holding the search-ready chunks, synced to Vector Search. |
| `VS_ENDPOINT_NAME` | `marketing_campaign_vs` | Vector Search endpoint name. |
| `INDEX_NAME` | `contract_chunks_index` | Vector Search index name (also the UC securable name). |
| `EMBEDDING_MODEL` | `databricks-gte-large-en` | Embedding model endpoint for managed embeddings. |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("VOLUME_NAME", "contracts", "VOLUME_NAME — UC volume with fictional_*.pdf contracts")
dbutils.widgets.text("SOURCE_TABLE", "contract_chunks", "SOURCE_TABLE — search-ready chunks table (CDF enabled)")
dbutils.widgets.text("VS_ENDPOINT_NAME", "marketing_campaign_vs", "VS_ENDPOINT_NAME — Vector Search endpoint name")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "INDEX_NAME — Vector Search index name")
dbutils.widgets.text("EMBEDDING_MODEL", "databricks-gte-large-en", "EMBEDDING_MODEL — managed embedding model endpoint")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
SOURCE_TABLE = dbutils.widgets.get("SOURCE_TABLE")
VS_ENDPOINT_NAME = dbutils.widgets.get("VS_ENDPOINT_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
EMBEDDING_MODEL = dbutils.widgets.get("EMBEDDING_MODEL")

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
SOURCE_TABLE_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE}"
INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"
print("Contracts volume :", VOLUME_PATH)
print("Chunks table     :", SOURCE_TABLE_FQN)
print("Index            :", INDEX_FQN)


### Parse + Chunk the Contracts in One Step

`ai_parse_document` reads each PDF; `ai_prep_search` turns the parsed layout
into RAG-ready chunks — automatically splitting content, dropping boilerplate,
and enriching each chunk with document context (title, section, page, etc.).

The result is a clean Delta table with one row per chunk:
- `id` — unique chunk id (used as the Vector Search primary key)
- `chunk_to_embed` — context-enriched text we embed for similarity search
- `chunk_to_retrieve` — raw chunk text returned to the agent at query time
- `source_uri` — the contract file name

Change Data Feed (CDF) is enabled so the Delta Sync index can track updates.


In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}')

# If the PDFs were landed by Lakeflow Connect (0_lakeflow-connect), the volume holds them
# under generated ids; the connector's contract_files table maps each one back to its
# original file name. Otherwise the volume file name is already the contract name.
CONTRACT_FILES_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.contract_files"
if spark.catalog.tableExists(CONTRACT_FILES_FQN):
    files_source = f"""
    SELECT coalesce(m._file_metadata.file_name, regexp_extract(f.path, '([^/]+)$', 1)) AS file_name,
           f.content
    FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile') f
    LEFT JOIN {CONTRACT_FILES_FQN} m ON m.file.uri = f.path"""
else:
    files_source = f"""
    SELECT regexp_extract(path, '([^/]+)$', 1) AS file_name, content
    FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile')"""

spark.sql(f"""
CREATE OR REPLACE TABLE {SOURCE_TABLE_FQN} AS
WITH files AS ({files_source}
),
parsed_documents AS (
  SELECT file_name, ai_parse_document(content) AS parsed
  FROM files
),
prepped_documents AS (
  SELECT file_name, ai_prep_search(parsed) AS result
  FROM parsed_documents
)
SELECT
  chunk.value:chunk_id::STRING              AS id,
  chunk.value:chunk_position::INT           AS chunk_position,
  chunk.value:chunk_to_retrieve::STRING     AS chunk_to_retrieve,
  chunk.value:chunk_to_embed::STRING        AS chunk_to_embed,
  file_name                                 AS source_uri
FROM prepped_documents,
     LATERAL variant_explode(prepped_documents.result:document.contents) AS chunk
""")

spark.sql(f'ALTER TABLE {SOURCE_TABLE_FQN} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

cnt = spark.table(SOURCE_TABLE_FQN).count()
print(f'Wrote {cnt} search-ready chunks to {SOURCE_TABLE_FQN}')
display(spark.sql(f'SELECT source_uri, chunk_position, chunk_to_retrieve FROM {SOURCE_TABLE_FQN} ORDER BY source_uri, chunk_position LIMIT 8'))

### Create the Vector Search Endpoint

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, PipelineType,
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn,
)

w = WorkspaceClient()

existing = [e.name for e in w.vector_search_endpoints.list_endpoints()]
if VS_ENDPOINT_NAME in existing:
    print('Reusing endpoint', VS_ENDPOINT_NAME)
else:
    print('Creating endpoint', VS_ENDPOINT_NAME, '(may take a few minutes)...')
    w.vector_search_endpoints.create_endpoint_and_wait(name=VS_ENDPOINT_NAME, endpoint_type=EndpointType.STANDARD)
    print('Endpoint online.')


### Create (or Sync) the Delta Sync Index

In [ ]:
import time

def index_exists(fqn):
    try:
        w.vector_search_indexes.get_index(index_name=fqn); return True
    except Exception:
        return False

if index_exists(INDEX_FQN):
    print('Index exists — triggering sync')
    w.vector_search_indexes.sync_index(index_name=INDEX_FQN)
else:
    w.vector_search_indexes.create_index(
        name=INDEX_FQN, endpoint_name=VS_ENDPOINT_NAME, primary_key='id',
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE_FQN, pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[EmbeddingSourceColumn(name='chunk_to_embed', embedding_model_endpoint_name=EMBEDDING_MODEL)],
        ),
    )
    print('Created index', INDEX_FQN)


### Wait for the Index to Finish its Sync

In [ ]:
deadline = time.time() + 60 * 25
while True:
    idx = w.vector_search_indexes.get_index(index_name=INDEX_FQN)
    ready = getattr(idx.status, 'ready', None)
    rows_indexed = getattr(idx.status, 'indexed_row_count', None)
    print(f'ready={ready} indexed_rows={rows_indexed}')
    if ready:
        break
    if time.time() > deadline:
        print('Still syncing; it is safe to continue once ready=True.'); break
    time.sleep(30)


### Result

In [ ]:
print('Vector Search endpoint :', VS_ENDPOINT_NAME)
print('Vector Search index    :', INDEX_FQN)
print('Embedding model        :', EMBEDDING_MODEL)
print('Embedding column       : chunk_to_embed (context-enriched by ai_prep_search)')
